In [1]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd, pyarrow.parquet as pq, joblib
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().parent                  # the nids folder
sys.path.insert(0, str(ROOT))
from src.features import feature_columns, to_matrix
from src.evaluation import evaluate

SEED = 11
out_dir = ROOT / "data" / "processed"
all_cols = pq.ParquetFile(out_dir / "train.parquet").schema.names
cols = feature_columns(all_cols)

train = pd.read_parquet(out_dir / "train.parquet", columns=cols)
train = train.sample(500_000, random_state=SEED)       # subsample for speed
print(train.shape)

(500000, 40)


In [2]:
x_raw = to_matrix(train, cols)
keep = x_raw.std(axis=0) > 0
cols = [c for c, k in zip(cols, keep) if k]
print("features used:", len(cols), "| dropped constant:", int((~keep).sum()))

scaler = StandardScaler().fit(x_raw[:, keep])
x_train = scaler.transform(x_raw[:, keep]).astype(np.float32)
(ROOT / "results").mkdir(exist_ok=True)
joblib.dump({"cols": cols, "scaler": scaler}, ROOT / "results" / "scaler.joblib")

features used: 40 | dropped constant: 0


['c:\\Users\\HP\\Desktop\\Ejust\\Projects\\Data Security\\nids\\results\\scaler.joblib']

In [3]:
model = IsolationForest(n_estimators=200, random_state=SEED, n_jobs=-1)
model.fit(x_train)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",200
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary <n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",11
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",'auto'
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary <warm_start>`... versionadded:: 0.21",False
Name,Type,Value
estimator_ estimator_: :class:`~sklearn.tree.ExtraTreeRegressor` instanceThe child estimator template used to create the collection offitted sub-estimators... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,ExtraTreeRegressor,ExtraTreeRegr...ndom_state=11)


In [4]:
def iter_split(name, cap=None, seed=SEED):
    pf = pq.ParquetFile(out_dir / f"{name}.parquet")
    rng = np.random.default_rng(seed)
    keep_p = {}
    if cap:
        vc = pd.read_parquet(out_dir / f"{name}.parquet", columns=["Attack"])["Attack"].value_counts()
        keep_p = {k: min(1.0, cap / v) for k, v in vc.items() if k != "Benign"}
    for batch in pf.iter_batches(batch_size=500_000, columns=cols + ["row_id", "Attack"]):
        df = batch.to_pandas()
        if keep_p:
            p = df["Attack"].map(keep_p).fillna(1.0).to_numpy(dtype=float)
            df = df[rng.random(len(df)) < p]
        yield df

def score_split(name, cap=None):
    parts = []
    for df in iter_split(name, cap):
        x = scaler.transform(to_matrix(df, cols)).astype(np.float32)
        s = -model.score_samples(x)                  # higher = more anomalous
        parts.append(pd.DataFrame({
            "id": df["row_id"].to_numpy(), "score": s,
            "label": (df["Attack"] != "Benign").astype(int).to_numpy(),
            "attack_type": df["Attack"].astype(str).to_numpy(), "split": name}))
    return pd.concat(parts, ignore_index=True)

In [5]:
val = score_split("val")
test = score_split("test", cap=100_000)     # all benign, at most 100k per attack type
print(len(val), len(test))

res = evaluate(test, val_df=val)
print(json.dumps(res, indent=2))

(ROOT / "results" / "scores").mkdir(parents=True, exist_ok=True)
pd.concat([val, test]).to_parquet(ROOT / "results" / "scores" / f"m1_iforest_nftoniot_seed{SEED}.parquet")

539110 1758003
{
  "auroc": 0.5199164122592747,
  "aupr": 0.3854364210738246,
  "tpr@fpr0.001": 0.07390470970445445,
  "real_fpr@0.001": 0.041827379014564965,
  "tpr@fpr0.01": 0.10045266316029311,
  "real_fpr@0.01": 0.07341575960027742,
  "tpr@fpr0.05": 0.5634341853895437,
  "real_fpr@0.05": 0.5704203173434869,
  "per_attack_tpr@fpr0.01": {
    "backdoor": 0.0037479921470640726,
    "ddos": 0.1330458730350179,
    "dos": 0.00047823531169982766,
    "injection": 0.08291479776029598,
    "mitm": 0.04033440891757113,
    "password": 0.3372566230450048,
    "ransomware": 0.09964622641509434,
    "scanning": 0.02608634791129242,
    "xss": 0.043304643714971974
  }
}


In [6]:
from sklearn.metrics import roc_auc_score

b = test[test["label"] == 0]

# 1. Do val and test benign scores differ? (0.5 = same distribution)
d = pd.concat([val.assign(g=0), b.assign(g=1)])
print("val-vs-test benign AUROC:", round(roc_auc_score(d["g"], d["score"]), 3))

qs = [0.5, 0.9, 0.99, 0.999]
print(pd.DataFrame({"val_benign": val["score"].quantile(qs),
                    "test_benign": b["score"].quantile(qs)}))

# 2. AUROC per attack type against test benign (below 0.5 means ranked as MORE normal than benign)
for t, g in test[test["label"] == 1].groupby("attack_type"):
    dd = pd.concat([b, g])
    print(f"{t:12s} AUROC {roc_auc_score(dd['label'], dd['score']):.3f}  n={len(g)}")

# 3. Does the tune split, which is closer in time to val, behave better?
tune = score_split("tune")
r = evaluate(tune, val_df=val)
print({k: round(r[k], 3) for k in ["auroc", "real_fpr@0.01", "real_fpr@0.05"]})

val-vs-test benign AUROC: 0.927
       val_benign  test_benign
0.500    0.378655     0.720088
0.900    0.645270     0.765738
0.990    0.775027     0.811560
0.999    0.786374     0.816574
backdoor     AUROC 0.554  n=16809
ddos         AUROC 0.642  n=99492
dos          AUROC 0.231  n=100369
injection    AUROC 0.759  n=100549
mitm         AUROC 0.398  n=6818
password     AUROC 0.761  n=100256
ransomware   AUROC 0.700  n=3392
scanning     AUROC 0.193  n=100014
xss          AUROC 0.529  n=99920
{'auroc': 0.806, 'real_fpr@0.01': 0.011, 'real_fpr@0.05': 0.102}


In [7]:
# A. where in the test split does benign drift start?
tb = test[test["label"] == 0].sort_values("id").reset_index(drop=True)
tb["part"] = pd.qcut(tb.index, 8, labels=False)
print(tb.groupby("part").agg(first_id=("id", "min"), median_score=("score", "median")))
print("val median:", round(val["score"].median(), 3))

# B. which features shifted? (in train-standard-deviation units)
def benign_sample(name, n=200_000):
    df = pd.concat([d[d["Attack"] == "Benign"] for d in iter_split(name)])
    return df.sample(min(n, len(df)), random_state=SEED)

bv, bt_ = benign_sample("val"), benign_sample("test")
xv = scaler.transform(to_matrix(bv, cols))
xt = scaler.transform(to_matrix(bt_, cols))
shift = pd.Series(xt.mean(0) - xv.mean(0), index=cols)
print(shift.reindex(shift.abs().sort_values(ascending=False).index).head(10))

# C. per-attack AUROC on tune (benign has not drifted there)
tb2 = tune[tune["label"] == 0]
for name, g in tune[tune["label"] == 1].groupby("attack_type"):
    dd = pd.concat([tb2, g])
    print(f"{name:12s} tune AUROC {roc_auc_score(dd['label'], dd['score']):.3f}  n={len(g)}")

      first_id  median_score
part                        
0      8758262      0.700523
1      8968990      0.719820
2     10570017      0.722259
3     12559569      0.747691
4     13724787      0.743378
5     14565728      0.719600
6     15608065      0.731061
7     16415926      0.710717
val median: 0.379
DURATION_OUT              6.644722
RETRANSMITTED_IN_PKTS     5.700068
RETRANSMITTED_IN_BYTES    5.015000
DURATION_IN               3.538903
IN_BYTES                  3.239160
LONGEST_FLOW_PKT          3.043556
MAX_IP_PKT_LEN            3.043556
SHORTEST_FLOW_PKT         2.830071
MIN_TTL                   2.692547
MAX_TTL                   2.672558
dtype: float32
ddos         tune AUROC 0.927  n=4284
dos          tune AUROC 0.861  n=2537
injection    tune AUROC 0.957  n=4602
mitm         tune AUROC 0.870  n=63
password     tune AUROC 0.964  n=7138
scanning     tune AUROC 0.798  n=363461
xss          tune AUROC 0.869  n=1874


In [8]:
tune = score_split("tune")
pd.concat([val, tune, test]).to_parquet(
    ROOT / "results" / "scores" / f"m1_iforest_nftoniot_seed{SEED}.parquet")